# w9_cv.ipynb — 5-fold CV, ZS-ONLY @2000ep (paper main table)

Refactor (user): ZS-only (no FT head; --head off), EPOCHS=2000 to match the
fixed-split budget, VRAM multi-tower scheduler (like w9_flash -- warmup
measures the cap-512 CV cost once, then packs recipe x fold jobs per GPU by
measured budget). Selection is the user's noname-focused metric:
**cvsel = noname_hit@1 + noname_hit@5 + 2 x noname_tagF1** (on each fold's VAL
set), written to zsbest_w9cv_<recipe>_fold<k>_fp.json. Full-pool protocol
(same as the fixed split). AUTO-STOPS when the 5x|recipes| grid is drained.


In [ ]:
# w9_cv.ipynb -- constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"

RECIPES = [                      # core comparison (user): I-CE vs CE
    "wcle_ce_cetf",              # discriminative baseline (I = 0)
    "wcle_i2ce_icetf",           # I-CE (CE + I x2)
]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 2000, 50, 2, 10   # ZS: seeds unused

# VRAM scheduler knobs (like w9_flash): budget = SAFETY x free VRAM - RESERVE.
SAFETY = 0.85
RESERVE_GIB = 1.5
os.makedirs(OUT_DIR, exist_ok=True)
print("jobs :", len(RECIPES) * N_FOLDS, f"({len(RECIPES)} recipes x {N_FOLDS} folds) @ {EPOCHS}ep")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# VRAM-BUDGET MULTI-TOWER SCHEDULER (like w9_flash): warmup measures the
# cap-512 CV cost once (real step, incl backward), then packs recipe x fold
# jobs per GPU by budget. ZS-only "done" = zsbest_<nm>_fp.json present; a
# partial/crashed fold auto-continues from its newest checkpoint.
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] free/GPU ~{min(free.values()) / 2**30:.0f}G  budgets "
      f"{[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")

# warmup: one cap-512 CV measure (i2ce-ish recipe, carries I)
mrec = RECIPES[1] if len(RECIPES) > 1 else RECIPES[0]
tf = Path(tempfile.gettempdir()) / "w9cv_vram.txt"
tf.unlink(missing_ok=True)
cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir", MEAS_OUT,
       "--repo", REPO, "--arm", mrec, "--fold", "0", "--n-folds", str(N_FOLDS),
       "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
       "--measure-vram", str(tf)]
print(f"[warmup] measuring CV cap512 via {mrec} ...", flush=True)
with open(logd / "measure_cv.log", "w") as fh:
    subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
COST = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
print(f"[warmup] CV cap512 cost {COST / 2**30:.2f}G -> "
      f"~{min(budget[g] // COST for g in gpus)} towers/GPU", flush=True)

todo = []
for r in RECIPES:
    for k in range(N_FOLDS):
        nm = J.cv_label(r, k)
        if (Path(OUT_DIR) / f"zsbest_{nm}_fp.json").exists():
            print(f"[skip] {nm} done"); continue
        todo.append((r, k, nm))

now_used = {g: 0 for g in gpus}
fails = []
cvn = threading.Condition()

def run_job(g, r, k, nm):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--epochs", str(EPOCHS),
               "--ckpt-every", str(CKPT_EVERY), "--ckpt-seeds", str(CKPT_SEEDS),
               "--topup-seeds", str(TOPUP_SEEDS), "--full-pool",
               "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{r}_fold{k}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 60:.1f}m]", flush=True)
    finally:
        with cvn:
            now_used[g] -= COST
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, k, nm = pending[i]
            fit = [g for g in gpus if now_used[g] + COST <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += COST
            th = threading.Thread(target=run_job, args=(g, r, k, nm), daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} (used {now_used[g] / 2**30:.1f}"
                  f"/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"CV drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: per-recipe 5-fold mean +- std, ZS-primary (zsbest by cvsel).
import json
import numpy as np
from pathlib import Path

def _load(nm):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    return json.loads(p.read_text()) if p.exists() else None

print(f"{'recipe':26s} folds  non@1         non@5         tag_non       "
      f"cvsel  neu@1")
for r in RECIPES:
    rows = [d for d in (_load(J.cv_label(r, k)) for k in range(N_FOLDS)) if d]
    lab = r.replace("wcle_", "").replace("_icetf", "").replace("_cetf", "")
    if not rows:
        print(f"{lab:26s} 0/{N_FOLDS} (none)"); continue
    def ms(key):
        v = [d[key] for d in rows]; return np.mean(v), np.std(v)
    n1, n5, tg, cs, nu = (ms("nm_noname"), ms("h5_noname"), ms("tag_noname"),
                          ms("cvsel"), ms("nm_neutral"))
    print(f"{lab:26s} {len(rows)}/{N_FOLDS}  {n1[0]:.3f}+-{n1[1]:.3f}  "
          f"{n5[0]:.3f}+-{n5[1]:.3f}  {tg[0]:.3f}+-{tg[1]:.3f}  "
          f"{cs[0]:.2f}   {nu[0]:.3f}")


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)